# Predict Chemical Reaction

## SMILES -> Graph

In [1]:
%pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.0/30.0 MB 11.0 MB/s  0:00:02 11.0 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.Chem import PandasTools
import pandas as pd

In [5]:
def smiles_to_mol(smiles):
    """
    Convert a SMILES string into an RDKit Mol object.
    Returns None if the SMILES string is invalid.
    """
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None

    return mol


def print_mol_basic_info(smiles):
    """
    Print basic RDKit molecule information.
    """
    mol = smiles_to_mol(smiles)

    if mol is None:
        return

    canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

    print("Original SMILES:", smiles)
    print("Canonical SMILES:", canonical_smiles)
    print("Number of atoms:", mol.GetNumAtoms())
    print("Number of bonds:", mol.GetNumBonds())

    return mol

In [ ]:
reactant_1 = "CCO"    # ethanol
reactant_2 = "O=O"    # oxygen

mol_1 = print_mol_basic_info(reactant_1)
print()
mol_2 = print_mol_basic_info(reactant_2)

=== Atoms / Nodes ===
{'idx': 0, 'symbol': 'C', 'atomic_num': 6, 'degree': 1, 'formal_charge': 0, 'is_aromatic': False, 'num_hydrogens': 3}
{'idx': 1, 'symbol': 'C', 'atomic_num': 6, 'degree': 3, 'formal_charge': 0, 'is_aromatic': False, 'num_hydrogens': 0}
{'idx': 2, 'symbol': 'O', 'atomic_num': 8, 'degree': 1, 'formal_charge': 0, 'is_aromatic': False, 'num_hydrogens': 0}
{'idx': 3, 'symbol': 'C', 'atomic_num': 6, 'degree': 3, 'formal_charge': 0, 'is_aromatic': True, 'num_hydrogens': 0}
{'idx': 4, 'symbol': 'C', 'atomic_num': 6, 'degree': 2, 'formal_charge': 0, 'is_aromatic': True, 'num_hydrogens': 1}
{'idx': 5, 'symbol': 'C', 'atomic_num': 6, 'degree': 2, 'formal_charge': 0, 'is_aromatic': True, 'num_hydrogens': 1}
{'idx': 6, 'symbol': 'C', 'atomic_num': 6, 'degree': 2, 'formal_charge': 0, 'is_aromatic': True, 'num_hydrogens': 1}
{'idx': 7, 'symbol': 'C', 'atomic_num': 6, 'degree': 2, 'formal_charge': 0, 'is_aromatic': True, 'num_hydrogens': 1}
{'idx': 8, 'symbol': 'C', 'atomic_num':

## Data Stucture
- Atoms
- Edges_index (Atoms - Atoms)
- Edges_attr - bond feature matrix

In [ ]:
from rdkit import Chem
import torch
from torch_geometric.data import Data


def atom_features(atom):
    return [
        atom.GetAtomicNum(),          # 원자 번호: C=6, O=8
        atom.GetDegree(),             # 연결된 atom 수
        atom.GetFormalCharge(),       # formal charge
        int(atom.GetIsAromatic()),    # aromatic 여부
        atom.GetTotalNumHs(),         # 붙어있는 H 개수
    ]


def bond_features(bond):
    bond_type = bond.GetBondType()

    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ]


def mol_to_pyg_data(mol):
    # 1. atom → node feature matrix x
    x = []
    for atom in mol.GetAtoms():
        x.append(atom_features(atom))

    x = torch.tensor(x, dtype=torch.float)

    # 2. bond → edge_index + edge_attr
    edge_index = []
    edge_attr = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        feat = bond_features(bond)

        # chemical bond는 undirected이므로 양방향 edge 추가
        edge_index.append([i, j])
        edge_attr.append(feat)

        edge_index.append([j, i])
        edge_attr.append(feat)

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    return data